# Modelado - Enfoque A: Validación Estática (Hold-out)
En esta notebook utilizaremos la estrategia clásica de validación estática, donde hemos separado un 70% para entrenamiento y un 15% para validación (dejando el 15% final para test).

In [ ]:
import math
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Modelos Baseline
from sklearn.dummy import DummyClassifier

# Modelos Lineales
from sklearn.linear_model import LogisticRegression

# Árboles y Ensambles
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Análisis Discriminante
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Naive Bayes
from sklearn.naive_bayes import GaussianNB

# Vecinos más cercanos (k-NN)
from sklearn.neighbors import KNeighborsClassifier

# Redes Neuronales
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score, precision_score, f1_score, recall_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


### 1. Carga de Datos

In [ ]:
base_path = '../data/model_input/static/'

X_train = pd.read_csv(base_path + 'X_train.csv')
y_train = pd.read_csv(base_path + 'y_train.csv').values.ravel()

X_val = pd.read_csv(base_path + 'X_val.csv')
y_val = pd.read_csv(base_path + 'y_val.csv').values.ravel()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

### 2. Definición del Preprocesador (ColumnTransformer)

In [ ]:
vars_categoricas = [
    'meal', 'market_segment', 'distribution_channel', 
    'reserved_room_type', 'deposit_type', 'customer_type'
]

vars_numericas = [
    'lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 
    'previous_cancellations', 'required_car_parking_spaces', 
    'total_of_special_requests', 'agent', 'company', 
    'arrival_date_year', 'arrival_month_num' 
]

vars_binarias = [
    'is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

### 3. Definimos los modelos que vamos a entrenar y el pipeline completo

In [ ]:
# --- 1. DEFINICIÓN DE MODELOS ---
RANDOM_STATE = 42

modelos = {
    "Baseline (mayoría)": DummyClassifier(strategy="most_frequent"),
    
    # --- Modelos CON class_weight='balanced' ---
    "Regresión Logística": LogisticRegression(C=1, max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
    "Árbol de Decisión": DecisionTreeClassifier(max_depth=12, random_state=RANDOM_STATE, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'),
    "Gradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    
    # --- Modelos SIN class_weight en sklearn ---
    "LDA": LinearDiscriminantAnalysis(),
    "QDA (regularizado)": QuadraticDiscriminantAnalysis(reg_param=0.3),
    "Naive Bayes": GaussianNB(),
    "k-NN (k=15)": KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    "Red Neuronal (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=RANDOM_STATE, early_stopping=True),
}

pipelines_entrenados = {}
resultados_metricas = []

# --- 2. PREPARAR EL GRÁFICO PARA LAS MATRICES ---
n_modelos = len(modelos)
cols = 3  # 3 columnas de gráficos
rows = math.ceil(n_modelos / cols) # Calcula las filas necesarias automáticamente

# Crear la figura y los "ejes" (subplots)
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
axes = axes.flatten() # Aplanamos el array de ejes para iterar fácilmente

# --- 3. BUCLE DE ENTRENAMIENTO Y EVALUACIÓN ---
for i, (nombre, modelo) in enumerate(modelos.items()):
    print(f"Entrenando: {nombre}...")
    
    # Crear y entrenar pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', modelo)
    ])
    pipeline.fit(X_train, y_train)
    pipelines_entrenados[nombre] = pipeline
    
    # Predicciones
    y_pred = pipeline.predict(X_val)
    
    # Métricas
    if hasattr(pipeline, "predict_proba"):
        y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
        roc_auc = roc_auc_score(y_val, y_pred_proba)
    else:
        roc_auc = None
        
    resultados_metricas.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1-Score": f1_score(y_val, y_pred, zero_division=0),
        "ROC-AUC": roc_auc
    })
    
    # Generar la matriz de confusión y dibujarla en su subplot correspondiente (axes[i])
    cm = confusion_matrix(y_val, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Canc.', 'Canc.'])
    
    # colorbar=False para que no se llene de barras de colores repetidas
    disp.plot(cmap=plt.cm.Blues, ax=axes[i], colorbar=False) 
    axes[i].set_title(nombre, fontsize=12, fontweight='bold')

# Ocultar los gráficos vacíos (si sobran espacios en la cuadrícula)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Ajustar el espaciado y mostrar las matrices
plt.tight_layout()
print("\nGenerando matrices de confusión...")
plt.show()

# --- 4. MOSTRAR LA TABLA COMPARATIVA ---
df_resultados = pd.DataFrame(resultados_metricas)
df_resultados = df_resultados.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("TABLA COMPARATIVA DE MODELOS (Ordenada por F1-Score)")
print("="*70)
display(df_resultados) # Si usas script normal en vez de Jupyter, cambia display() por print()

Para mejorar el rendimiento de tus modelos (especialmente buscando un buen equilibrio entre Precisión y Recall para tu problema de reservas de hotel), te recomiendo ajustar los siguientes hiperparámetros clave mediante **GridSearchCV** o **RandomizedSearchCV**. 

Aquí tienes los modelos principales y qué hiperparámetros vale la pena cambiarles:

### 1. HistGradientBoostingClassifier (Suele ser el mejor modelo)
Este modelo es extremadamente potente, pero por defecto puede sobreajustarse o no aprender lo suficiente dependiendo de los datos.
* **`learning_rate`**: (Ej. `0.01`, `0.05`, `0.1`, `0.2`). Controla qué tan rápido aprende el modelo. Si lo bajas, necesitas aumentar `max_iter`.
* **`max_iter`**: (Ej. `100`, `200`, `500`). Es el equivalente a `n_estimators`.
* **`max_depth`**: (Ej. `3`, `5`, `9`, `None`). Los modelos de Boosting suelen funcionar mejor con árboles poco profundos (max_depth entre 3 y 7).
* **`l2_regularization`**: (Ej. `0.0`, `0.1`, `1.0`). Ayuda a penalizar hojas extremas y reduce el sobreajuste.

### 2. RandomForestClassifier
Ya viste el impacto gigante que tiene `max_depth`. Para afinarlo sin que se disparen los falsos positivos:
* **`max_depth`**: (Ej. `15`, `20`, `25`, `None`). Busca el punto dulce donde el Recall sea alto pero la Precisión no caiga al 50%.
* **`min_samples_leaf`**: (Ej. `1`, `5`, `10`). Exige que cada "hoja" final tenga un mínimo de muestras. Esto es **excelente** para evitar que el modelo memorice casos raros y suaviza las predicciones.
* **`max_features`**: (Ej. `'sqrt'`, `'log2'`, `0.5`). Controla cuántas variables mira cada árbol. Cambiarlo a un porcentaje (como 0.5, es decir, el 50% de las variables) a veces mejora mucho el rendimiento.

### 3. Regresión Logística
Es un modelo lineal, por lo que su ajuste es más sencillo pero muy importante para evitar que coeficientes irrelevantes arruinen la predicción.
* **`C`**: (Ej. `0.01`, `0.1`, `1`, `10`). Es la fuerza de la regularización (inversa: valores más bajos = más regularización). Si tienes muchas variables creadas (OneHotEncoding), bajar el valor de `C` ayuda a ignorar el ruido.
* **`penalty`**: (Ej. `'l2'`, `'l1'`). Si usas el solver `liblinear` o `saga`, puedes usar `'l1'`, lo cual forzará a que las variables inútiles tengan un peso de exactamente 0 (selección de características automática).

### 4. k-NN (K-Nearest Neighbors)
* **`n_neighbors`**: (Ej. `5`, `15`, `30`, `50`). 15 está bien, pero en datasets grandes a veces números más altos (30-50) generalizan mejor.
* **`weights`**: (Ej. `'uniform'`, `'distance'`). Cambiarlo a `'distance'` hace que los vecinos más cercanos tengan más peso en la decisión que los lejanos. Esto suele mejorar mucho el k-NN en datasets desbalanceados.

### 5. Red Neuronal (MLPClassifier)
Las redes neuronales son muy sensibles a los hiperparámetros.
* **`hidden_layer_sizes`**: (Ej. `(64, 32)`, `(128, 64, 32)`, `(32,)`). Probar arquitecturas más profundas o más simples.
* **`alpha`**: (Ej. `0.0001`, `0.001`, `0.01`). Es la penalización L2. Aumentarla ayuda a que la red no se sobreajuste a los datos de entrenamiento.
* **`learning_rate_init`**: (Ej. `0.001`, `0.01`). Si la red no está convergiendo (termina muy rápido o muy lento), ajustar esto es clave.

### 💡 Mi sugerencia de plan de acción:
No intentes optimizar todos a la vez. Toma los **2 o 3 mejores modelos** de tu tabla comparativa (probablemente Random Forest, Gradient Boosting y quizás Regresión Logística por su interpretabilidad) y haz un `GridSearchCV / RandomizedSearchCV` solo sobre esos modelos con los rangos que te mencioné arriba.